# CYTools Interface

This notebook explains the boundary between StringForge and CYTools.  StringForge does not replace CYTools and does not try to duplicate toric-geometry computations.  Instead, it stores catalogue rows and cached derived data in a form that can be handed back to CYTools, or forwarded to JAXVacua when the goal is flux-vacuum physics.


## What belongs where?

| Task | Preferred tool |
| --- | --- |
| Query the public TDF catalogue, cache parquet shards, and run offline/HPC workflows. | StringForge |
| Reconstruct or inspect toric objects such as polytopes, triangulations, and cones. | CYTools |
| Build LCS periods, EFTs, and flux-vacuum finders from the stored data. | StringForge bridge into JAXVacua |

Use CYTools directly when you want to perform new toric-geometry computations.  Use StringForge when you want reproducible database access, cache management, and stable handoff to downstream physics packages.


## 1. Query a toric row

The pure database layer is enough for catalogue queries and polytope recovery.  No JAXVacua model is constructed in this step.


In [ ]:
from stringforge import TDFDatabase, LCSDatabase

raw_db = TDFDatabase()
rows = raw_db.query(h11=2).head(3)
rows[[c for c in ("h11", "h12", "ks_id", "triang_id") if c in rows.columns]]


## 2. Recover polytope data where supported

For TDF rows, `get_polytope` returns the stored polytope points and any additional polytope-level fields present in the parquet split.  This is the StringForge side of the handoff: it retrieves data reproducibly and leaves any further toric computations to CYTools.


In [ ]:
row = rows.iloc[0]
polytope_data = raw_db.get_polytope(
    ks_id=int(row["ks_id"]),
    h11=int(row["h11"]),
    h12=int(row["h12"]),
)
polytope_data.keys()


If CYTools is installed, the stored points can be passed to CYTools explicitly.  The import is kept optional because StringForge's database and vault workflows should remain usable without CYTools.


In [ ]:
# Optional CYTools handoff:
# from cytools import Polytope
# polytope = Polytope(polytope_data["polytope_points"])
# polytope.points()


## 3. Hand the same row to JAXVacua

When the next step is flux-vacuum physics rather than toric inspection, use `LCSDatabase`.  This layer presents Hodge numbers in the mirror convention expected by JAXVacua and can construct either an `lcs_tree` or a `FluxVacuaFinder`.


In [ ]:
lcs_db = LCSDatabase(dataset="tdf")

tree = lcs_db.load(
    ks_id=int(row["ks_id"]),
    triang_id=int(row["triang_id"]),
    # raw_db uses catalogue convention; LCSDatabase uses mirror convention.
    h11=int(row["h12"]),
    h12=int(row["h11"]),
    include_gv=False,
    include_conifolds=False,
)

type(tree).__name__, tree.h11, tree.h12


## Further reading

- Use the database tutorial for shard caching, batch loading, and offline/HPC operation.
- Use the JAXVacua CYTools and period tutorials for detailed physics workflows built from CYTools geometries.
- Use this notebook only as the interface map: StringForge retrieves and standardises data, CYTools computes toric geometry, and JAXVacua performs flux-vacuum physics.
